# In-Class Exercises 4 (solution)

The data that you will use come from the [replication package]() to Ricardo Duque
Gabriel, Mathias Klein, and Ana Sofia Pessoa: *The Political Costs of Austerity*,
forthcoming at the Review of Economics and Statistics.

> **Note:**
> 
> Please commit every time you solve one of the exercises. An example commit message
> could be `"Solution to question 1"`. Feel free to commit more than once per
> exercise if solving it requires multiple complicated steps.
>
> Push every now and then and switch to somebody else's machine.

## Using the `pathlib` library

---
### Question 1

Assign the path of the current directory to a variable `this_dir`. Verify that the type
of the variable is `pathlib.PosixPath` or `pathlib.WindowsPath`. Display the absolute
path of the directory.

In [ ]:

from pathlib import Path

this_dir = Path()
type(this_dir)

In [ ]:

this_dir.resolve()

---
### Question 2

In the `original_data` directory, there is a file called `data.txt`. Assign the path of
this file to a variable `data_file`. The type of the variable should be 
`pathlib.PosixPath` or `pathlib.WindowsPath`. 

- When creating the `Path` object, do not use absolute paths.
- Display the absolute path of the `data_file`.

In [ ]:

data_file = this_dir / "original_data" / "Data_Elections.dta"
data_file.resolve()

---
### Question 3 (spend 5 Minutes max on it, else consider as Bonus!)

Using only the objects `this_dir` and `data_file` along with their methods, get the relative path to `data_file` as seen from `this_dir`. Display the relative path.

> Note: We have not seen this in the screencast; you are on your own with your favourite
> search engine.

In [ ]:

data_file.resolve().relative_to(this_dir.resolve())

## pandas

---
### Question 4

- Import the `pandas` library as `pd`. 
- Set the options so you use "modern" Pandas as described in the first screencast. 
- Set the plotting backend to `plotly`.


In [ ]:

import pandas as pd

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = False
pd.options.plotting.backend = "plotly"


---
### Question 5

Read the file `Data_Elections.dta` into a `pd.DataFrame` object called `data`. Use the
`data_file` object for doing so.

You are likely to get an error when doing so. Find out
how to fix it. *(Hint: The error message is even more explicit than usually in Python;
you may want to have a carefully look at it despite its length. If working in VS Code,
make sure that you can see the entire message by selecting from the view options at the
bottom of the cell output)*

In [ ]:

data = pd.read_stata(data_file, convert_categoricals=False)

---
### Question 6

Familiarise yourself with the dataset. E.g., yo may want to look the column names, the
shape, some rows, data types ...

In [ ]:

data.columns

In [ ]:

data.shape

In [ ]:

data

In [ ]:

data.dtypes

---
### Question 7

We had to discard some information when reading the dataset. Luckily, we can access all
of it using a low-level `pd.StataReader` object. This will allow us to look at all meta
information that is stored with the dataset in Stata format.

In [ ]:
data_info = pd.io.stata.StataReader(data_file)

Look at the various `labels` attributes of the `pd.StataReader` object. Explain what
they do. Can you explain now why the error occurred when reading the dataset?

In [ ]:

data_info.variable_labels()

In [ ]:

data_info.value_labels()

In [ ]:

data_info.data_label


- `data_label` is a string describing the dataset. In this case, it is empty.
- `variable_labels` is a dictionary mapping the column names we have in our dataset to
  verbose descriptions.
- `value_labels` is a nested dictionary with column names as keys on the outer level.
  The inner dictionaries map the numeric values in the dataset to verbose descriptions.
  It is filled only for `ElectionType`. This is very much how `pd.Categorical` is
  stored internally, although we cannot see the numerical values in that case (which is
  a good thing! Far too easy to run numerical calculations with unordered categorical
  variables in Stata accidentally).
- The error occurred because the `National + Regional` label is repeated for two
  different values of `ElectionType`. While `value_labels` are supposed to be a
  bijection, Stata does not enforce this. Pandas will check it when converting the
  `value_labels` dictionary to a `pd.Categorical` object with the value labels as
  categories. Since categories need to be unique, 5 and 7 would be merged and
  information would be lost. Hence the error.


---
### Question 8

Look at the structure of `data` again. Would you keep all of

- `Country` and `cid`?
- `Nuts_id`,  `Name`, and `id`?

Why or why not?



- No reason to keep `cid`. Countries are very few and we have the country names. Most
  certainly, these are no official country codes, so we won't need them for merging.
- `Nuts_id` is a unique identifier for the regions and they look quite official. While
  we do have the names, they may easily differ across datasets (unicode characters, 
  parentheses, ...) or not be present in all datasets. `id` looks like a numerical code
  for `Nuts_id`, but we better verify.

---
### Question 9

Make sure that we can safely drop `cid` and `id` from the dataset by finding out the
unique combinations of the sets of variables from the previous question.

> Note: The `.unique()` method only works on Series, you'll need to find something else.

In [ ]:

data[["Country", "cid"]].drop_duplicates()

In [ ]:

(
    len(data[["Country", "cid"]].drop_duplicates())
    == len(data[["Country"]].drop_duplicates())
    == len(data[["cid"]].drop_duplicates())
)

In [ ]:

data[["Nuts_id", "Name", "id"]].drop_duplicates()

In [ ]:

(
    len(data[["Nuts_id", "Name", "id"]].drop_duplicates())
    == len(data["Nuts_id"].drop_duplicates())
    == len(data[["Name"]].drop_duplicates())
    == len(data[["id"]].drop_duplicates())
)

---
### Question 10

Drop `cid` and `id` from the dataset, which you should continue to store in the variable
`data`. Verify that the columns are gone.

To do so, you can either use the `drop(columns=[...])` method or select those columns
that you want to keep (there are even more options). Can you think of a reasons for each
strategy, particularly in an interactive setting like a notebook?

In [ ]:
data = data.drop(columns=["cid", "id"])
data.shape


- `drop()` is somewhat more explicit, particularly when the list of columns to be kept
  is long
- Selecting the columns to be kept makes it possible to execute the cell again without
  error (though non-linear execution is somewhat dangerous)

No clear winner here.

---
### Question 11

Give the columns (more) sensible names using the `lowercase_with_underscores`
convention. You should continue to store the dataset in the variable `data`. 

> Note: While trying out, you don't want to assign to `data` yet because once you change
> the column names, you cannot access the old ones anymore. That will eventually be 
> fine, but not in the interactive setting yet.

> Tip: You can just copy the output of the `variable_labels()` call above to get
> started.

In [ ]:

data = data.rename(
    columns={
        "Country": "country",
        "Nuts_id": "nuts_id",
        "Name": "nuts_name",
        "Year": "year",
        "ElectionType": "election_type",
        "EligibleVoters": "number_eligible_voters",
        "Valid": "number_valid_votes",
        "HHI": "number_parties_effective",
        "Far_Right": "number_votes_far_right",
        "Far_Left": "number_votes_far_left",
        "Far_Right_share": "share_votes_far_right",
        "Far_Left_share": "share_votes_far_left",
        "Far_share": "share_votes_far_any",
        "Turnout": "share_voter_turnout",
        "F0Far_Incumbent": "number_votes_far_any_incumbent",
        "left": "pm_party_left",
    },
)

---
### Question 12

Convert all variables to sensible data types. In some cases, you may want to keep the
column names, in some cases you may want to generate new ones. Briefly explain all of 
your choices.

If you want to set up any categorical variables, consider the following
1. Sometimes you may be able to use `Series.astype(pd.CategoricalDtype())`
1. However, make sure they have sensible categories
1. It might or might not be most readable to set up the categorical data type first
1. If you want to convert to float to an categorical with sensible labels, it will be
   easiest to do so in two steps. First convert to a `pd.IntXXDtype()` using, e.g.,
   `Series.astype('pd.Int64Dtype()')` and then convert to a categorical. This is due to
   "old" pandas dtypes, which could represent missing values only in floating points.
1. In any case, do not type (or copy/paste) any of the categories yourself, rather access
   (suitably converted) existing objects.
1. The exception to 4. is the duplicated value label above. Call one outcome
   `National + Regional A` and the other `National + Regional B`. Verify that you did
   not modify the original dictionary from `data_info`.
1. If you want to convert floating points to integers, you might get errors. Do not be
   discouraged, but find out why this happens and how to fix it. *(Hint: Make sure to
   use one of the nullable integer types we have seen, but that won't be enough)*


In [ ]:

for col in "country", "nuts_id", "nuts_name":
    data[col] = data[col].astype(pd.CategoricalDtype())

data["nuts_id"]


- All three columns already have sensible names and sensible outcomes
- We just need to convert to `pd.CategoricalDtype()`
- We verify for one of the three

In [ ]:

data["year"] = data["year"].astype(pd.Int16Dtype())
data["year"]

In [ ]:

election_cats = data_info.value_labels()["ElectionType"].copy()
for i, to_append in (5, " A"), (7, " B"):
    election_cats[i] += to_append
election_cats

In [ ]:

election_type = (
    data["election_type"].astype(pd.Int8Dtype()).astype(pd.CategoricalDtype())
)
election_type = election_type.cat.rename_categories(election_cats)
data["election_type"] = election_type
data["election_type"]


- First get Stata's `value_labels` and fix the duplicate. The copy is important, else
  we would modify the original dictionary. Note that it is important to make the copy
  of the inner dictionary, not the outer one.
- Election type first needs to be converted to a nullable Integer data type to match
  the contents. Which length we use does not matter, this is temporary, anyhow.
- Then can convert to `category`, can verify at that point that categories are 
  {1, 2, ..., 7}.
- Then we call `.rename_categories()` with the fixed dictionary and verify the result.

In [ ]:

for col in data.columns:
    if col.startswith("number_") and col != "number_parties_effective":
        data[col] = data[col].round().astype(pd.UInt32Dtype())
data["number_eligible_voters"]


- We can loop over the columns since we have given them sensible names, just need to be
  careful with the Herfindahl-Hirschman index (could debate that choice of column name)
- Apparently some of the values are numerically too far away from integers so that
  pandas complains. We need to explicitly round first. In cases where it would be
  crucial to get the correct integer, we may want to investigate deeper if that happens
  (e.g., we expect integers 1 to 7, but may have a 1.4 in there). Here, it does not
  matter given the size of the electorates and likely measurement error.
- We use `UInt32`, but it really does not matter as long as we can represent all
  numbers. Seems like a good idea to disallow negative numbers as a safety measure.

In [ ]:

pm_party_orientation = (
    data["pm_party_left"].astype(pd.Int8Dtype()).astype(pd.CategoricalDtype())
)
pm_party_orientation = pm_party_orientation.cat.rename_categories(
    {0: "Other", 1: "Left-leaning"},
)
data["pm_party_orientation"] = pm_party_orientation
data = data.drop(columns="pm_party_left")
pm_party_orientation


- Even though we often say "X is a dummy variable for whatever", the better
  representation usually are categorical variables.
- This will make plotting etc. easier and more consistent.
- Decent statistical packages handle these out-of-the-box, too

---
### Question 13

Summarise the cleaned data in a similar way as above. Now also look at summary
statistics, including value counts of categorical variables. Do you notice anything
when doing the latter? If so, will you have to be careful for interpreting descriptive
statistics?

In [ ]:

data.columns

In [ ]:

data.shape

In [ ]:

data

In [ ]:

data.dtypes

In [ ]:

data.describe()

In [ ]:

data["country"].value_counts()

In [ ]:

data["nuts_id"].value_counts().unique()

In [ ]:

data["election_type"].value_counts()

In [ ]:

data["pm_party_orientation"].value_counts()


- Value counts for NUTS ids show that we have a balanced panel
- Apparently always last election results are used, see query below
- Need to be extremely careful when doing descriptives, because all other
  variables are filled.

In [ ]:

data.query("nuts_id == 'SE33'")

---
### Question 14

Make three plots of mean vote shares by year, across all NUTS regions and elections.
- far right share
- far left share
- far share (any)

Don't worry about whether these plots make a lot of sense (i.e., any weighting with
electorate size or the like).

In [ ]:

only_elections = data[data["election_type"].notna()]
only_elections.shape

In [ ]:

only_elections.groupby("year")["share_votes_far_right"].mean().plot()

In [ ]:

only_elections.groupby("year")["share_votes_far_left"].mean().plot()

In [ ]:

only_elections.groupby("year")["share_votes_far_any"].mean().plot()

---
### Question 15

Make three scatterplots of vote shares by year, across all NUTS regions and elections.

- far right share
- far left share
- far share (any)

Colour the dots using the country.

In [ ]:

only_elections.plot.scatter(x="year", y="share_votes_far_right", color="country")

In [ ]:

only_elections.plot.scatter(x="year", y="share_votes_far_left", color="country")

In [ ]:

only_elections.plot.scatter(x="year", y="share_votes_far_any", color="country")